In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

In [2]:
groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = [f"{group}_{t}" for group in groups for t in range(8)]
columns.append("target")

df = pd.read_csv(
    "TomsHardware.data",
    header=None,
    names=columns
)

df["target_log"] = np.log1p(df["target"])
print(df.shape)
#df.head(20)

(28179, 98)


In [8]:
df.head()

,NCD_0,NCD_1,NCD_2,NCD_3,NCD_4,NCD_5,NCD_6,NCD_7,BL_0,BL_1,...,AS_NAC_0,AS_NAC_1,AS_NAC_2,AS_NAC_3,AS_NAC_4,AS_NAC_5,AS_NAC_6,AS_NAC_7,target,target_log
0,0,0,0,0,0,0,0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.5,1.704748
1,0,0,0,0,0,0,0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.5,1.504077
2,0,0,0,0,0,0,0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.098612
3,0,0,0,0,0,0,0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.098612
4,0,0,0,0,0,0,0,0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.5,0.916291


In [9]:
X = df.drop(["target", "target_log"] + [col for col in df.columns if col.startswith("ND_")],axis=1)
y = df["target_log"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

In [13]:
y_pred = np.clip(y_pred, 0, None)
y_pred_real = np.expm1(y_pred)
y_test_real = np.expm1(y_test)

In [14]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
mae = mean_absolute_error(y_test_real, y_pred_real)
r2 = r2_score(y_test_real, y_pred_real)

print(rmse, mae, r2)

18206778.099232916 409813.8994806947 -1685108.9374875948


In [15]:
# 9. Results

print("\nMetrics in log-scale:")
print(f"RMSE: {rmse_log:.4f}")
print(f"MAE:  {mae_log:.4f}")
print(f"R²:   {r2_log:.4f}")

print("\nMetrics in original scale:")
print(f"RMSE: {rmse_real:.4f}")
print(f"MAE:  {mae_real:.4f}")
print(f"R²:   {r2_real:.4f}")


Metrics in log-scale:


NameError: name 'rmse_log' is not defined

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# === Конфігурація ===
groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]
columns = [f"{group}_{t}" for group in groups for t in range(8)]
columns.append("target")

# === Завантаження даних ===
df = pd.read_csv(
    "TomsHardware.data",
    header=None,
    names=columns
)

print(f"Розмір датасету: {df.shape}")
print(f"Пропущені значення: {df.isnull().sum().sum()}")

# === Логарифмічне перетворення ===
df["target_log"] = np.log1p(df["target"])

# === Ознаки та ціль ===
drop_cols = ["target", "target_log"] + [col for col in df.columns if col.startswith("ND_")]
X = df.drop(drop_cols, axis=1)
y = df["target_log"]

print(f"\nКількість ознак: {X.shape[1]}")
print(f"Розподіл target: min={df['target'].min():.1f}, max={df['target'].max():.1f}, mean={df['target'].mean():.1f}")

# === Розбивка на train/test ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# === Масштабування ===
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# === Функція оцінки моделі ===
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    y_pred_log = model.predict(X_te)
    y_pred_log = np.clip(y_pred_log, 0, None)

    # Метрики в log просторі
    r2_log = r2_score(y_te, y_pred_log)

    # Зворотне перетворення
    y_pred_real = np.expm1(y_pred_log)
    y_test_real = np.expm1(y_te)

    rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
    mae = mean_absolute_error(y_test_real, y_pred_real)
    r2_real = r2_score(y_test_real, y_pred_real)

    print(f"\n{'='*40}")
    print(f"Модель: {name}")
    print(f"  R² (log простір):  {r2_log:.4f}")
    print(f"  R² (реальний):     {r2_real:.4f}")
    print(f"  RMSE:              {rmse:.2f}")
    print(f"  MAE:               {mae:.2f}")

    return r2_log, r2_real

# === Порівняння моделей ===
print("\n=== ПОРІВНЯННЯ МОДЕЛЕЙ ===")

# 1. Лінійна регресія (оригінальна)
evaluate_model(
    "Лінійна регресія",
    LinearRegression(),
    X_train_scaled, X_test_scaled,
    y_train, y_test
)

# 2. Random Forest (не потребує масштабування)
evaluate_model(
    "Random Forest",
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    X_train, X_test,
    y_train, y_test
)

# 3. Gradient Boosting
evaluate_model(
    "Gradient Boosting",
    GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
    X_train, X_test,
    y_train, y_test
)

print("\n=== ГОТОВО ===")
print("Якщо R² в log просторі теж поганий — проблема в даних або ознаках.")
print("Якщо R² в log просторі нормальний — проблема у викидах при зворотному перетворенні.")

Розмір датасету: (28179, 97)
Пропущені значення: 0

Кількість ознак: 88
Розподіл target: min=0.0, max=265916.5, mean=3486.4

=== ПОРІВНЯННЯ МОДЕЛЕЙ ===

Модель: Лінійна регресія
  R² (log простір):  0.6240
  R² (реальний):     -1685108.9375
  RMSE:              18206778.10
  MAE:               409813.90

Модель: Random Forest
  R² (log простір):  0.7690
  R² (реальний):     0.8936
  RMSE:              4575.93
  MAE:               1082.96

Модель: Gradient Boosting
  R² (log простір):  0.7399
  R² (реальний):     0.8533
  RMSE:              5372.55
  MAE:               1213.67

=== ГОТОВО ===
Якщо R² в log просторі теж поганий — проблема в даних або ознаках.
Якщо R² в log просторі нормальний — проблема у викидах при зворотному перетворенні.


In [16]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Підготовка target
df["target_log"] = np.log1p(df["target"])

# 2. Формуємо X і y
# Прибираємо original target, log target і ND_ фічі
X = df.drop(
    ["target", "target_log"] + [col for col in df.columns if col.startswith("ND_")],
    axis=1
)
y = df["target_log"]

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# 6. Predictions in log-space
y_pred_log = lr_model.predict(X_test_scaled)

# 7. Metrics in log-space
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
mae_log = mean_absolute_error(y_test, y_pred_log)
r2_log = r2_score(y_test, y_pred_log)

# 8. Results
print("\nMetrics in log-scale:")
print(f"RMSE: {rmse_log:.4f}")
print(f"MAE:  {mae_log:.4f}")
print(f"R²:   {r2_log:.4f}")


Metrics in log-scale:
RMSE: 1.4585
MAE:  1.1422
R²:   0.6223
